# 15. Capstone: The `oracle` Prototype

## Background

Every notebook in this series has built toward one practical skill: **making better decisions under uncertainty**. We started with Bayes' theorem and conjugate updates, moved through MCMC samplers, hierarchical models, GLMs, model comparison, Gaussian processes, and variational inference. In the final stretch we operationalized these ideas — posterior predictive checks to catch model failure, ROPE for decision regions, A/B testing with Beta-Binomial posteriors, and sequential stopping rules that respect optional stopping.

The capstone brings it together. `oracle` is a small but complete Bayesian experimentation library — the kind of thing a data team would actually deploy to run product experiments. It is not a toy: it handles real concerns like minimum sample burn-in, asymmetric business loss functions, three independent stopping rules, and human-readable reports.

## What You'll Learn

- How to package Bayesian A/B logic into a clean, reusable class
- Composing the full inference pipeline: conjugate update → posterior metrics → ROPE decision → expected loss → stopping rule
- Asymmetric loss functions and how to express business priorities in probabilistic terms
- Building a `report()` method that translates posteriors into language stakeholders understand
- End-to-end simulation: running `oracle` on a live experiment stream and auditing its decisions

## Why This Matters

Most teams ship A/B tests with frequentist tooling that was designed for agriculture experiments in the 1920s. They set α = 0.05, run until significance, and claim a winner — a workflow that inflates false positive rates, ignores effect size, and discards business context entirely.

`oracle` replaces this with a coherent decision framework: stop when you have enough evidence that the difference is practically meaningful (ROPE-HDI rule), or when the probability of being wrong falls below an acceptable business loss threshold. No p-value hacking, no peeking inflation, no binary significance theater.

The community consensus (Kruschke 2018, Deng & Shi 2016, Bayesian A/B testing at Etsy/Booking.com) is that sequential Bayesian testing with ROPE is strictly superior for product experimentation: it stops earlier on average, controls false positives without correction, and surfaces effect size alongside the go/no-go signal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.special import betaln
from dataclasses import dataclass, field
from typing import Optional, Callable, Literal
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Core Data Structures

`oracle` tracks experiment state as a small dataclass so every intermediate quantity is inspectable.

In [ ]:
@dataclass
class ExperimentState:
    """Sufficient statistics for a Beta-Binomial experiment arm."""
    name: str
    alpha_prior: float = 1.0   # Beta(alpha, beta) prior
    beta_prior: float = 1.0
    n_obs: int = 0             # observations added so far
    k_obs: int = 0             # successes

    @property
    def alpha_post(self) -> float:
        return self.alpha_prior + self.k_obs

    @property
    def beta_post(self) -> float:
        return self.beta_prior + (self.n_obs - self.k_obs)

    @property
    def posterior_mean(self) -> float:
        return self.alpha_post / (self.alpha_post + self.beta_post)

    @property
    def posterior_std(self) -> float:
        a, b = self.alpha_post, self.beta_post
        return np.sqrt(a * b / ((a + b)**2 * (a + b + 1)))

    def sample(self, size: int = 50_000) -> np.ndarray:
        return np.random.beta(self.alpha_post, self.beta_post, size)


@dataclass
class Decision:
    """Result of oracle.decide()."""
    rule_triggered: Optional[str]   # which stopping rule fired, or None
    should_stop: bool
    recommendation: str             # 'control' | 'treatment' | 'equivalent' | 'continue'
    p_trt_better: float
    expected_loss_ctrl: float
    expected_loss_trt: float
    rope_verdict: str               # 'equivalent' | 'different' | 'inconclusive'
    hdi_lo: float
    hdi_hi: float
    delta_mean: float


ctrl_state = ExperimentState('control', alpha_prior=2, beta_prior=8)
ctrl_state.n_obs, ctrl_state.k_obs = 100, 12
print(f"Control: posterior mean={ctrl_state.posterior_mean:.3f}, std={ctrl_state.posterior_std:.3f}")
print(f"  Beta({ctrl_state.alpha_post}, {ctrl_state.beta_post})")

## 2. Utility Functions

These are the same primitives developed in notebooks 12-14, now packaged cleanly.

In [ ]:
def hdi(samples: np.ndarray, prob: float = 0.95) -> tuple[float, float]:
    """Highest Density Interval via sorted-window method. O(n log n)."""
    sorted_samples = np.sort(samples)
    n = len(sorted_samples)
    window = int(np.floor(prob * n))
    widths = sorted_samples[window:] - sorted_samples[:n - window]
    idx = np.argmin(widths)
    return float(sorted_samples[idx]), float(sorted_samples[idx + window])


def rope_decision(
    samples: np.ndarray,
    rope_lo: float,
    rope_hi: float,
    hdi_prob: float = 0.95
) -> str:
    """Three-way ROPE decision based on 95% HDI overlap with ROPE."""
    lo, hi = hdi(samples, hdi_prob)
    entirely_in_rope = (lo >= rope_lo) and (hi <= rope_hi)
    no_overlap = (hi < rope_lo) or (lo > rope_hi)
    if entirely_in_rope:
        return 'equivalent'
    elif no_overlap:
        return 'different'
    else:
        return 'inconclusive'


def expected_loss(
    samples_a: np.ndarray,
    samples_b: np.ndarray,
    loss_fn: Optional[Callable] = None
) -> tuple[float, float]:
    """
    Expected loss for choosing A vs choosing B.

    Default loss: E[max(0, theta_b - theta_a)] for choosing A (i.e., cost of
    choosing A when B is actually better, proportional to how much better B is).
    """
    if loss_fn is None:
        loss_fn = lambda a, b: np.maximum(0, b - a)

    el_choose_a = np.mean(loss_fn(samples_a, samples_b))  # loss when we pick A
    el_choose_b = np.mean(loss_fn(samples_b, samples_a))  # loss when we pick B
    return float(el_choose_a), float(el_choose_b)


# Quick sanity check
s_ctrl = np.random.beta(2, 18, 50_000)   # ~10% conversion
s_trt  = np.random.beta(3, 17, 50_000)   # ~15% conversion
lo, hi = hdi(s_trt - s_ctrl)
print(f"Delta HDI: [{lo:.3f}, {hi:.3f}]")
print(f"ROPE decision (-0.01, 0.01): {rope_decision(s_trt - s_ctrl, -0.01, 0.01)}")
el_a, el_b = expected_loss(s_ctrl, s_trt)
print(f"Expected loss choosing ctrl={el_a:.4f}, trt={el_b:.4f}")

## 3. The `oracle` Class

The full experiment engine. Three stopping rules, composable loss functions, incremental updates.

In [ ]:
class oracle:
    """
    Bayesian A/B testing engine.

    Parameters
    ----------
    alpha_prior, beta_prior : float
        Shared Beta prior for both arms (default: uniform Beta(1,1)).
    rope : tuple[float, float]
        Region of Practical Equivalence for the conversion rate difference.
        Effects within ROPE are treated as practically zero.
    min_n : int
        Minimum observations per arm before any stopping rule fires.
    loss_threshold : float
        Stop if min(expected_loss_ctrl, expected_loss_trt) < loss_threshold.
    p_threshold : float
        Stop if P(trt > ctrl) > p_threshold or < (1 - p_threshold).
    hdi_prob : float
        HDI credible mass for ROPE and interval reporting.
    n_samples : int
        Monte Carlo draws for posterior calculations.
    loss_fn : callable, optional
        Custom loss function loss_fn(chosen_rate, other_rate) → loss.
        Default: E[max(0, other - chosen)].

    Usage
    -----
    exp = oracle(rope=(-0.01, 0.01), min_n=200)
    exp.update(n_ctrl=500, k_ctrl=48, n_trt=500, k_trt=63)
    decision = exp.decide()
    exp.report()
    """

    def __init__(
        self,
        alpha_prior: float = 1.0,
        beta_prior: float = 1.0,
        rope: tuple[float, float] = (-0.01, 0.01),
        min_n: int = 100,
        loss_threshold: float = 0.001,
        p_threshold: float = 0.95,
        hdi_prob: float = 0.95,
        n_samples: int = 50_000,
        loss_fn: Optional[Callable] = None,
    ):
        self.ctrl = ExperimentState('control', alpha_prior, beta_prior)
        self.trt  = ExperimentState('treatment', alpha_prior, beta_prior)
        self.rope = rope
        self.min_n = min_n
        self.loss_threshold = loss_threshold
        self.p_threshold = p_threshold
        self.hdi_prob = hdi_prob
        self.n_samples = n_samples
        self.loss_fn = loss_fn

        # History for sequential monitoring
        self._history: list[dict] = []

    # ------------------------------------------------------------------
    # Data ingestion
    # ------------------------------------------------------------------

    def update(self, n_ctrl: int, k_ctrl: int, n_trt: int, k_trt: int) -> 'oracle':
        """Add new observations. Conjugate Beta update (O(1))."""
        self.ctrl.n_obs += n_ctrl
        self.ctrl.k_obs += k_ctrl
        self.trt.n_obs  += n_trt
        self.trt.k_obs  += k_trt
        return self   # fluent API

    # ------------------------------------------------------------------
    # Posterior quantities
    # ------------------------------------------------------------------

    def _draw_samples(self) -> tuple[np.ndarray, np.ndarray]:
        s_ctrl = self.ctrl.sample(self.n_samples)
        s_trt  = self.trt.sample(self.n_samples)
        return s_ctrl, s_trt

    def evaluate(self) -> dict:
        """Compute all posterior metrics. Returns a plain dict."""
        s_ctrl, s_trt = self._draw_samples()
        delta = s_trt - s_ctrl

        p_trt_better = float(np.mean(s_trt > s_ctrl))
        hdi_lo, hdi_hi = hdi(delta, self.hdi_prob)
        rope_verdict = rope_decision(delta, self.rope[0], self.rope[1], self.hdi_prob)
        el_ctrl, el_trt = expected_loss(s_ctrl, s_trt, self.loss_fn)

        return dict(
            n_ctrl=self.ctrl.n_obs,
            n_trt=self.trt.n_obs,
            ctrl_mean=self.ctrl.posterior_mean,
            trt_mean=self.trt.posterior_mean,
            delta_mean=float(np.mean(delta)),
            delta_std=float(np.std(delta)),
            p_trt_better=p_trt_better,
            hdi_lo=hdi_lo,
            hdi_hi=hdi_hi,
            rope_verdict=rope_verdict,
            el_ctrl=el_ctrl,
            el_trt=el_trt,
        )

    # ------------------------------------------------------------------
    # Decision logic
    # ------------------------------------------------------------------

    def decide(self) -> Decision:
        """
        Apply stopping rules in priority order:
          1. ROPE-HDI: HDI entirely within ROPE → equivalent, stop
          2. P-superiority: P(trt > ctrl) > threshold → treatment wins, stop
                            P(trt > ctrl) < 1-threshold → control wins, stop
          3. Expected loss: min(EL) < loss_threshold → stop (pick lower-loss arm)

        Returns Decision(should_stop=False) if min_n not yet reached.
        """
        metrics = self.evaluate()

        # Enforce minimum sample size
        if self.ctrl.n_obs < self.min_n or self.trt.n_obs < self.min_n:
            return Decision(
                rule_triggered=None, should_stop=False,
                recommendation='continue',
                p_trt_better=metrics['p_trt_better'],
                expected_loss_ctrl=metrics['el_ctrl'],
                expected_loss_trt=metrics['el_trt'],
                rope_verdict=metrics['rope_verdict'],
                hdi_lo=metrics['hdi_lo'], hdi_hi=metrics['hdi_hi'],
                delta_mean=metrics['delta_mean'],
            )

        rule = None
        recommendation = 'continue'

        # Rule 1: ROPE-HDI equivalence
        if metrics['rope_verdict'] == 'equivalent':
            rule = 'rope_hdi'
            recommendation = 'equivalent'

        # Rule 2: probability of superiority
        elif metrics['p_trt_better'] > self.p_threshold:
            rule = 'p_superiority'
            recommendation = 'treatment'
        elif metrics['p_trt_better'] < (1 - self.p_threshold):
            rule = 'p_superiority'
            recommendation = 'control'

        # Rule 3: expected loss
        elif min(metrics['el_ctrl'], metrics['el_trt']) < self.loss_threshold:
            rule = 'expected_loss'
            recommendation = 'treatment' if metrics['el_trt'] < metrics['el_ctrl'] else 'control'

        should_stop = rule is not None

        # Log to history
        self._history.append({**metrics, 'rule': rule, 'recommendation': recommendation})

        return Decision(
            rule_triggered=rule,
            should_stop=should_stop,
            recommendation=recommendation,
            p_trt_better=metrics['p_trt_better'],
            expected_loss_ctrl=metrics['el_ctrl'],
            expected_loss_trt=metrics['el_trt'],
            rope_verdict=metrics['rope_verdict'],
            hdi_lo=metrics['hdi_lo'],
            hdi_hi=metrics['hdi_hi'],
            delta_mean=metrics['delta_mean'],
        )

    # ------------------------------------------------------------------
    # Human-readable report
    # ------------------------------------------------------------------

    def report(self) -> None:
        """Print a stakeholder-friendly experiment summary."""
        d = self.decide()
        m = self.evaluate()

        status_icon = {
            'treatment': 'SHIP IT',
            'control':   'HOLD — control wins',
            'equivalent': 'CALL IT — no meaningful difference',
            'continue':  'STILL RUNNING',
        }[d.recommendation]

        print("=" * 60)
        print(f" oracle Experiment Report")
        print("=" * 60)
        print(f" Status : {status_icon}")
        if d.rule_triggered:
            print(f" Stopped by : {d.rule_triggered} rule")
        print()
        print(f" Observations")
        print(f"   Control   : n={m['n_ctrl']:,}  conversion={m['ctrl_mean']:.3%}")
        print(f"   Treatment : n={m['n_trt']:,}  conversion={m['trt_mean']:.3%}")
        print()
        print(f" Effect Estimate (treatment − control)")
        lift_pct = m['delta_mean'] / m['ctrl_mean'] * 100 if m['ctrl_mean'] else 0
        print(f"   Mean delta : {m['delta_mean']:+.4f}  ({lift_pct:+.1f}% relative lift)")
        print(f"   {int(self.hdi_prob*100)}% HDI  : [{m['hdi_lo']:+.4f}, {m['hdi_hi']:+.4f}]")
        print()
        print(f" Decision Metrics")
        print(f"   P(trt > ctrl)    : {d.p_trt_better:.3f}")
        print(f"   ROPE {self.rope}  : {d.rope_verdict}")
        print(f"   Expected loss if choose ctrl : {d.expected_loss_ctrl:.5f}")
        print(f"   Expected loss if choose trt  : {d.expected_loss_trt:.5f}")
        print("=" * 60)


# Quick demo
exp = oracle(rope=(-0.005, 0.005), min_n=50)
exp.update(n_ctrl=400, k_ctrl=40, n_trt=400, k_trt=56)
exp.report()

## 4. Sequential Monitoring Loop

`oracle` is designed for streaming data — call `.update()` each day as batches arrive, then `.decide()` to check stopping rules. Here we simulate an experiment where treatment truly has a 2pp lift.

In [ ]:
def run_experiment_stream(
    true_ctrl: float,
    true_trt: float,
    batch_size: int = 50,
    max_batches: int = 60,
    rope: tuple[float, float] = (-0.005, 0.005),
    min_n: int = 200,
    seed: int = 42,
) -> tuple[oracle, list[dict]]:
    """Simulate streaming experiment data through oracle."""
    rng = np.random.default_rng(seed)
    exp = oracle(rope=rope, min_n=min_n, loss_threshold=5e-4, p_threshold=0.95)
    log = []

    for batch in range(max_batches):
        # Simulate one day of traffic
        ctrl_outcomes = rng.binomial(batch_size, true_ctrl)
        trt_outcomes  = rng.binomial(batch_size, true_trt)
        exp.update(batch_size, int(ctrl_outcomes), batch_size, int(trt_outcomes))

        decision = exp.decide()
        metrics  = exp.evaluate()

        log.append({
            'batch': batch + 1,
            'n_ctrl': exp.ctrl.n_obs,
            'p_trt_better': metrics['p_trt_better'],
            'delta_mean': metrics['delta_mean'],
            'hdi_lo': metrics['hdi_lo'],
            'hdi_hi': metrics['hdi_hi'],
            'rope_verdict': metrics['rope_verdict'],
            'stopped': decision.should_stop,
            'rule': decision.rule_triggered,
            'recommendation': decision.recommendation,
        })

        if decision.should_stop:
            print(f"Stopped at batch {batch+1} "
                  f"(n={exp.ctrl.n_obs}/arm) — rule: {decision.rule_triggered}, "
                  f"recommend: {decision.recommendation}")
            break
    else:
        print(f"Reached max batches ({max_batches}) without stopping.")

    return exp, log


# Scenario A: real 2pp lift (true positive)
exp_a, log_a = run_experiment_stream(true_ctrl=0.10, true_trt=0.12, seed=1)

# Scenario B: no effect (test for false positive)
exp_b, log_b = run_experiment_stream(true_ctrl=0.10, true_trt=0.10, seed=2)

# Scenario C: negligible lift inside ROPE (practical equivalence)
exp_c, log_c = run_experiment_stream(true_ctrl=0.10, true_trt=0.102, seed=3)

## 5. Visualising the Sequential Decision Path

Track probability of superiority, delta HDI, and ROPE overlap over time for each scenario.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 11))
fig.suptitle('oracle Sequential Monitoring — Three Scenarios', fontsize=14, fontweight='bold')

scenarios = [
    (log_a, 'A: True 2pp lift (TP)', '#2196F3'),
    (log_b, 'B: No effect (FP check)', '#F44336'),
    (log_c, 'C: Negligible lift (Equivalence)', '#4CAF50'),
]

for row, (log, title, color) in enumerate(scenarios):
    batches = [e['batch'] for e in log]
    p_vals  = [e['p_trt_better'] for e in log]
    d_means = [e['delta_mean'] for e in log]
    hdi_lo  = [e['hdi_lo'] for e in log]
    hdi_hi  = [e['hdi_hi'] for e in log]

    stop_idx = next((i for i, e in enumerate(log) if e['stopped']), None)

    # Left: P(trt > ctrl)
    ax = axes[row, 0]
    ax.plot(batches, p_vals, color=color, lw=2)
    ax.axhline(0.95, color='#FF9800', ls='--', lw=1, label='P threshold (0.95)')
    ax.axhline(0.05, color='#FF9800', ls='--', lw=1)
    ax.axhline(0.5, color='gray', ls=':', lw=1)
    if stop_idx is not None:
        ax.axvline(batches[stop_idx], color='black', ls='--', lw=1.5, label='Stop point')
    ax.set_ylim(0, 1)
    ax.set_ylabel('P(trt > ctrl)')
    ax.set_title(title)
    ax.legend(fontsize=8)

    # Right: Delta HDI
    ax = axes[row, 1]
    ax.fill_between(batches, hdi_lo, hdi_hi, alpha=0.3, color=color, label='95% HDI')
    ax.plot(batches, d_means, color=color, lw=2, label='Posterior mean')
    ax.axhline(0, color='gray', ls='-', lw=1)
    ax.axhspan(-0.005, 0.005, alpha=0.15, color='#9C27B0', label='ROPE')
    if stop_idx is not None:
        ax.axvline(batches[stop_idx], color='black', ls='--', lw=1.5)
    ax.set_ylabel('Delta (trt − ctrl)')
    ax.set_title(f'{title} — Delta HDI')
    ax.legend(fontsize=8)

for ax in axes[-1]:
    ax.set_xlabel('Batch (day)')

plt.tight_layout()
plt.savefig('oracle_sequential_monitoring.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 6. Error Rate Calibration

How well does `oracle` control false positive and false negative rates across many simulated experiments?

In [ ]:
def calibration_study(
    n_sims: int = 500,
    true_lift: float = 0.0,       # set to 0 for FPR, >0 for FNR/power
    batch_size: int = 50,
    max_batches: int = 80,
    rope: tuple = (-0.005, 0.005),
    min_n: int = 150,
    seed_start: int = 0,
) -> dict:
    """Run many simulated experiments and tally decision outcomes."""
    true_ctrl = 0.10
    true_trt  = true_ctrl + true_lift

    results = {'stopped': 0, 'trt_wins': 0, 'ctrl_wins': 0,
               'equivalent': 0, 'no_stop': 0, 'stop_n': []}

    for seed in range(seed_start, seed_start + n_sims):
        rng = np.random.default_rng(seed)
        exp = oracle(rope=rope, min_n=min_n, loss_threshold=5e-4, p_threshold=0.95)
        final_rec = 'no_stop'

        for _ in range(max_batches):
            ctrl_out = int(rng.binomial(batch_size, true_ctrl))
            trt_out  = int(rng.binomial(batch_size, true_trt))
            exp.update(batch_size, ctrl_out, batch_size, trt_out)
            d = exp.decide()

            if d.should_stop:
                final_rec = d.recommendation
                results['stop_n'].append(exp.ctrl.n_obs)
                break

        results['stopped']   += (final_rec != 'no_stop')
        results['trt_wins']  += (final_rec == 'treatment')
        results['ctrl_wins'] += (final_rec == 'control')
        results['equivalent']+= (final_rec == 'equivalent')
        results['no_stop']   += (final_rec == 'no_stop')

    results['n_sims'] = n_sims
    results['pct_stopped']  = results['stopped'] / n_sims
    results['mean_stop_n']  = np.mean(results['stop_n']) if results['stop_n'] else None
    return results


print("Running calibration study (this takes ~30s)...")
# FPR: no true effect
fpr_study = calibration_study(n_sims=300, true_lift=0.0, seed_start=0)
# Power: 2pp lift
pwr_study = calibration_study(n_sims=300, true_lift=0.02, seed_start=1000)

print(f"\n=== Null hypothesis (no lift) ===")
print(f"  Experiments stopped     : {fpr_study['pct_stopped']:.1%}")
print(f"  Claimed 'treatment wins': {fpr_study['trt_wins']/300:.1%}  ← empirical FPR")
print(f"  Claimed 'equivalent'    : {fpr_study['equivalent']/300:.1%}")
print(f"  Claimed 'control wins'  : {fpr_study['ctrl_wins']/300:.1%}")
if fpr_study['mean_stop_n']:
    print(f"  Mean n/arm at stop      : {fpr_study['mean_stop_n']:.0f}")

print(f"\n=== Alternative hypothesis (+2pp lift) ===")
print(f"  Experiments stopped     : {pwr_study['pct_stopped']:.1%}")
print(f"  Claimed 'treatment wins': {pwr_study['trt_wins']/300:.1%}  ← empirical power")
print(f"  Claimed 'equivalent'    : {pwr_study['equivalent']/300:.1%}")
print(f"  Mean n/arm at stop      : {pwr_study['mean_stop_n']:.0f}" if pwr_study['mean_stop_n'] else "")

## 7. Asymmetric Business Loss

The default loss function treats over-promising and under-delivering symmetrically. Real businesses rarely have symmetric stakes. Shipping a degraded experience costs more than not shipping a small win.

We encode this by weighting the loss of shipping a worse variant more heavily than the opportunity cost of not shipping a slightly better one.

In [ ]:
def asymmetric_loss(cost_false_positive: float = 3.0):
    """
    Return a loss function that penalises false positives (shipping a
    worse variant) `cost_false_positive` times more than false negatives
    (missing a true improvement).

    oracle.expected_loss() calls loss_fn(chosen_rate, other_rate).
    A false positive occurs when we choose A but B > A.
    We want to penalise large negative deltas more.
    """
    def _loss(chosen: np.ndarray, other: np.ndarray) -> np.ndarray:
        diff = other - chosen   # positive ⟹ we picked the worse arm
        return np.where(diff > 0, cost_false_positive * diff, -diff * 0.5)
    return _loss


# Compare symmetric vs asymmetric loss on a close call
def compare_loss_functions(n_ctrl, k_ctrl, n_trt, k_trt):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Symmetric vs Asymmetric Loss — Effect on Decision Threshold', fontsize=13)

    for ax, (label, loss_fn) in zip(axes, [
        ('Symmetric (default)', None),
        ('Asymmetric (FP costs 3x)', asymmetric_loss(3.0)),
    ]):
        exp = oracle(rope=(-0.005, 0.005), min_n=10, loss_fn=loss_fn, p_threshold=0.95)
        exp.update(n_ctrl, k_ctrl, n_trt, k_trt)
        m = exp.evaluate()
        d = exp.decide()

        s_ctrl = exp.ctrl.sample(30_000)
        s_trt  = exp.trt.sample(30_000)
        delta  = s_trt - s_ctrl

        ax.hist(delta, bins=80, density=True, alpha=0.7, color='steelblue', label='Posterior Δ')
        lo, hi = hdi(delta)
        ax.axvspan(lo, hi, alpha=0.2, color='orange', label=f'95% HDI [{lo:.3f}, {hi:.3f}]')
        ax.axvspan(-0.005, 0.005, alpha=0.2, color='purple', label='ROPE')
        ax.axvline(0, color='gray', lw=1)

        status = f"Recommend: {d.recommendation}"
        el_ctrl, el_trt = d.expected_loss_ctrl, d.expected_loss_trt
        ax.set_title(f'{label}\n{status} | EL_ctrl={el_ctrl:.4f}, EL_trt={el_trt:.4f}')
        ax.set_xlabel('Δ conversion rate')
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('oracle_asymmetric_loss.png', dpi=120, bbox_inches='tight')
    plt.show()
    print("Figure saved.")


# Borderline experiment: small observed lift, close to ROPE
compare_loss_functions(n_ctrl=600, k_ctrl=62, n_trt=600, k_trt=74)

## 8. The Final Report

Run `oracle` on a realistic experiment and generate the stakeholder report.

In [ ]:
# Realistic checkout flow experiment
# Control: 8.2% conversion (historical baseline)
# Treatment: new checkout UX, targeting +1.5pp lift
# Traffic: 300 users/day per arm, 14-day runtime assumed
exp_final = oracle(
    rope=(-0.005, 0.005),
    min_n=1000,
    loss_threshold=2e-4,
    p_threshold=0.97,
    hdi_prob=0.95,
)

# Simulate 14 days of 300 users/arm
rng = np.random.default_rng(99)
for day in range(14):
    ctrl_day = int(rng.binomial(300, 0.082))
    trt_day  = int(rng.binomial(300, 0.097))
    exp_final.update(300, ctrl_day, 300, trt_day)

exp_final.report()

In [ ]:
# Visualise the full posterior for the final experiment
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('oracle Final Experiment — Posterior Summary', fontsize=13, fontweight='bold')

s_ctrl = exp_final.ctrl.sample(50_000)
s_trt  = exp_final.trt.sample(50_000)
delta  = s_trt - s_ctrl

# Panel 1: marginal posteriors
ax = axes[0]
ax.hist(s_ctrl, bins=60, density=True, alpha=0.6, color='#2196F3', label=f'Control (μ={exp_final.ctrl.posterior_mean:.3%})')
ax.hist(s_trt,  bins=60, density=True, alpha=0.6, color='#FF5722', label=f'Treatment (μ={exp_final.trt.posterior_mean:.3%})')
ax.set_xlabel('Conversion rate')
ax.set_ylabel('Density')
ax.set_title('Marginal Posteriors')
ax.legend()

# Panel 2: delta with HDI and ROPE
ax = axes[1]
lo, hi = hdi(delta)
ax.hist(delta, bins=80, density=True, alpha=0.75, color='#4CAF50')
ax.axvspan(lo, hi, alpha=0.25, color='orange', label=f'95% HDI [{lo:.3f}, {hi:.3f}]')
ax.axvspan(-0.005, 0.005, alpha=0.2, color='purple', label='ROPE (±0.5pp)')
ax.axvline(0, color='gray', lw=1)
ax.set_xlabel('Δ conversion (trt − ctrl)')
ax.set_title('Treatment Effect')
ax.legend(fontsize=9)

# Panel 3: cumulative P(trt > ctrl) over history
ax = axes[2]
n_history = range(1, exp_final.ctrl.n_obs + 1, 100)
p_sup = []
rng2 = np.random.default_rng(0)
# Rebuild history from scratch for the plot
a_c, b_c = exp_final.ctrl.alpha_prior, exp_final.ctrl.beta_prior
a_t, b_t = exp_final.trt.alpha_prior, exp_final.trt.beta_prior
# Use stored history if available, else just show current
if exp_final._history:
    hist_n = [h['n_ctrl'] for h in exp_final._history]
    hist_p = [h['p_trt_better'] for h in exp_final._history]
    ax.plot(hist_n, hist_p, color='#9C27B0', lw=2)
    ax.axhline(0.97, color='#FF9800', ls='--', lw=1.5, label='Stop threshold')
    ax.axhline(0.5, color='gray', ls=':', lw=1)
    ax.set_xlabel('N per arm')
    ax.set_ylabel('P(trt > ctrl)')
    ax.set_title('Monitoring Path')
    ax.set_ylim(0, 1)
    ax.legend()
else:
    ax.text(0.5, 0.5, 'No monitoring history
(no stopping rule fired)',
            ha='center', va='center', transform=ax.transAxes, fontsize=11)

plt.tight_layout()
plt.savefig('oracle_final_experiment.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Series Summary

You now have the full Bayesian inference toolkit:

| Notebook | Core Concept | Key Tool |
|----------|-------------|----------|
| 01 | Bayes' theorem & conjugate updates | Beta-Binomial, Gamma-Poisson |
| 02 | Priors: from ignorance to domain knowledge | `pm.Uniform`, `pm.HalfNormal`, `az.plot_dist` |
| 03 | MCMC: HMC & NUTS | `pm.sample`, convergence diagnostics |
| 04 | PyMC workflow | `pm.Data`, prior/posterior predictive checks |
| 05 | Hierarchical models | Non-centered parameterization, partial pooling |
| 06 | Bayesian GLMs | Logistic, Poisson, NegBin in PyMC |
| 07 | Model comparison | LOO-CV, WAIC, Pareto-k, stacking weights |
| 08 | Gaussian Processes | Kernel composition, `pm.gp.Marginal` |
| 09 | Variational Inference | ADVI, ELBO, mean-field vs full-rank |
| 10 | Bayesian Neural Networks | MC Dropout, BNN in PyMC, calibration |
| 11 | Posterior Predictive Checks | Misspecification diagnosis, LOO-PIT |
| 12 | ROPE & Decision Theory | HDI, three-way decision, asymmetric loss |
| 13 | Bayesian A/B Testing | Beta-Binomial, Thompson sampling |
| 14 | Sequential Testing | Stopping rules, optional stopping safety |
| **15** | **`oracle` Capstone** | **Full production Bayesian A/B engine** |

### The Core Loop

Every Bayesian analysis follows the same pattern:

```
1. Specify prior      →  what do we believe before seeing data?
2. Collect data       →  observe the world
3. Update posterior   →  Bayes' theorem / MCMC / VI
4. Check model        →  posterior predictive checks
5. Make decision      →  ROPE, expected loss, business framing
6. Iterate            →  new data → go to step 2
```

`oracle` automates steps 3, 4, and 5 for the Beta-Binomial case. The same architecture — streaming updates, ROPE decisions, expected loss — extends naturally to any parametric model where you can draw posterior samples.